# EasyRAG Storage And Workspace Artifacts

## Chapter position

This chapter turns prepared documents into retrieval infrastructure. Chunk boundaries, embedding inputs, vector storage, and workspace artifacts all start here.

## Learning question

Which files land in a workspace, and what does each storage layer actually own?

## Success criteria

- persist a local workspace to disk and inspect the files it writes
- compare in-memory stats with on-disk artifact names
- reopen the workspace and verify that the persisted artifacts still answer queries

## Source code anchors

- `easyrag.rag.orchestrator.EasyRAG.finalize_storages`
- `easyrag.rag.storage.local.JSONKVStorage`
- `easyrag.rag.storage.local.EmbeddingVectorStorage`
- `easyrag.rag.storage.local.NetworkXGraphStorage`


## Method principles

- `easyrag.rag.orchestrator.EasyRAG.finalize_storages`: This method persists local state and closes the lifecycle cleanly. In the notebooks it matters because temporary workspaces should be flushed before cleanup.
- `easyrag.rag.storage.local.JSONKVStorage`: This local backend owns persisted documents, chunks, summaries, and cache entries. It is the easiest storage layer to inspect directly on disk.
- `easyrag.rag.storage.local.EmbeddingVectorStorage`: This is the local dense-vector storage implementation. It stores embeddings and serves nearest-neighbor search, optionally with a better backend like HNSW when available.
- `easyrag.rag.storage.local.NetworkXGraphStorage`: This local graph backend stores entities, chunks, summaries, and relations as a navigable graph. It makes graph-aware retrieval and knowledge inspection possible without extra infrastructure.


## How the code fits together

The flow in this notebook is inserted corpus -> storage files -> reopened workspace -> aggregate stats. The goal is not to show every internal detail at once. The goal is to keep the boundary for this stage visible enough that later behavior still feels explainable. If a result looks odd, the intermediate objects in this notebook should tell you where to look next.

In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

## What this cell is proving

We first build a small local workspace, then explicitly finalize it so the JSON and NumPy artifacts are written to disk.

In [ ]:
artifact_tmp = tempfile.TemporaryDirectory()
artifact_rag = EasyRAG(
    working_dir=artifact_tmp.name,
    workspace="workspace-artifacts-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(artifact_rag.initialize_storages())
run_sync(
    artifact_rag.ainsert(
        [
            "# Architecture\nEasyRAG keeps repository guidance inspectable.\n",
            "# Retrieval\nQuery rewrite and grounded retrieval improve evidence coverage.\n",
        ],
        ids=["doc::architecture", "doc::retrieval"],
        file_paths=["docs/architecture.md", "docs/retrieval.md"],
    )
)
pre_finalize_stats = run_sync(artifact_rag.get_stats())
run_sync(artifact_rag.finalize_storages())
workspace_root = Path(artifact_tmp.name) / "workspace-artifacts-demo"
workspace_files = [
    str(path.relative_to(workspace_root))
    for path in sorted(workspace_root.rglob("*"))
    if path.is_file()
]
_print_json({"stats": pre_finalize_stats, "files": workspace_files})

## Why this output looks like this

The next cell opens a few of the persisted files directly. The goal is not to read every record. The goal is to confirm which artifact owns which kind of state.

In [ ]:
documents_json = json.loads((workspace_root / "kv" / "documents.json").read_text())
chunks_json = json.loads((workspace_root / "kv" / "chunks.json").read_text())
graph_json = json.loads((workspace_root / "graph.json").read_text())
relations_json = json.loads((workspace_root / "graph_relations.json").read_text())
status_json = json.loads((workspace_root / "doc_status.json").read_text())
_print_json(
    {
        "documents_keys": list(documents_json.keys())[:3],
        "chunk_keys": list(chunks_json.keys())[:3],
        "graph_top_level_keys": list(graph_json.keys())
        if isinstance(graph_json, dict)
        else type(graph_json).__name__,
        "relation_ids": list(relations_json.keys())[:3],
        "status_ids": list(status_json.keys())[:3],
    }
)

In [ ]:
reopened_rag = EasyRAG(
    working_dir=artifact_tmp.name,
    workspace="workspace-artifacts-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(reopened_rag.initialize_storages())
reopened_result = run_sync(
    reopened_rag.aquery(
        "How does EasyRAG improve evidence coverage?",
        QueryParam(
            mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=3
        ),
    )
)
_print_json(
    {"metadata": reopened_result.metadata, "citations": reopened_result.citations}
)
run_sync(reopened_rag.finalize_storages())

## What this cell is proving

The provider-backed path should preserve the same contract even when the backend behavior is less deterministic.

The optional path simply confirms that provider-backed workspaces still use the same artifact layout contract when they are configured to run locally.

In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(
        working_dir=provider_tmp.name, workspace="provider-workspace-artifacts-demo"
    )
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(
            provider_rag.ainsert(
                "# Retrieval\nGrounded retrieval stays inspectable.",
                ids=["doc::retrieval"],
                file_paths=["docs/retrieval.md"],
            )
        )
        run_sync(provider_rag.finalize_storages())
        provider_files = [
            str(
                path.relative_to(
                    Path(provider_tmp.name) / "provider-workspace-artifacts-demo"
                )
            )
            for path in sorted(
                (Path(provider_tmp.name) / "provider-workspace-artifacts-demo").rglob(
                    "*"
                )
            )
            if path.is_file()
        ]
        pprint(provider_files)
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        provider_tmp.cleanup()

## Common mistakes / debugging cues

- Do not tune retrieval before you understand chunk boundaries, embedding inputs, and backend choice.
- Watch metadata such as `chunk_strategy`, `vector_backend`, and workspace artifacts, not only final query hits.
- Indexing bugs often look like retrieval bugs until you inspect the payloads that were actually written.

## Takeaway

Finalization turns the in-memory workspace into a persistent teaching artifact. That is the point where documents, chunks, summaries, vectors, graph state, and document status stop being runtime-only objects and become inspectable files.

In [ ]:
artifact_tmp.cleanup()
print("Cleaned up the workspace-artifacts demo directory.")

## Where to go next

- Continue with [04_01_query_normalization_and_preprocessing.ipynb](../04_retrieval/04_01_query_normalization_and_preprocessing.ipynb) to follow those persisted artifacts into retrieval.
- Read [engineering/21-indexing-pipeline.md](../../docs/engineering/21-indexing-pipeline.md) for the code-oriented indexing walkthrough.
- Revisit [00_02_observing_rag_artifacts.ipynb](../00_overview/00_02_observing_rag_artifacts.ipynb) if you want the end-to-end artifact story again.